## Стенд моделирования цифровой линии связи

Один файл, весь код внутри. Модель тракта «передатчик → канал → приёмник»:
вероятность ошибки на бит, созвездие, спектр, глазковая диаграмма, проверка
разрядностей для цифровой реализации.

Зависимости: `numpy`, `scipy`, `pandas`, `plotly`.

Работа: правите раздел 1, выполняете всё (`Cell → Run All`), смотрите
разделы 5–9.

### 1. Параметры

Единственное место, которое нужно менять.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from scipy.special import erfc

MOD = "QPSK"          # "BPSK" | "QPSK"
RB = 1.0e6            # битовая скорость, бит/с
NBITS = 100_000       # бит за прогон (для BER 1e-3 хватает с запасом)
SPS = 8               # отсчётов на символ
ALPHA = 0.35          # скругление спектра (RRC)
SPAN = 8              # полуширина фильтра в символах

ESN0_DB = 10.0        # энергия символа к спектральной плотности шума
CFO_HZ = 0.0          # расстройка несущей, Гц
PHASE_DEG = 0.0       # начальный набег фазы, градусы
TIMING_ERR = 0.0      # ошибка момента стробирования, в символах
SEED = 42             # зерно генератора битов
CH_SEED = 7           # зерно шума

SWEEP = [4, 6, 8, 10, 12]   # точки свипа по Es/N0, дБ
SWEEP_BITS = 40_000         # бит на точку свипа

### 2. Передатчик

Биты → символы (код Грей) → формирующий фильтр RRC.

Почему RRC, а не прямоугольный импульс: прямоугольник даёт медленно
спадающий спектр (sinc) и большое внеполосное излучение. Фильтр со
скруглением спектра ограничивает полосу и при этом не создаёт межсимвольных
помех в моменты стробирования — если такой же фильтр стоит в приёмнике.

In [2]:
def make_bits(n, seed=SEED):
    """Случайные биты."""
    return np.random.default_rng(seed).integers(0, 2, n)


def rrc_taps(alpha, sps, span):
    """
    Импульсная характеристика фильтра RRC, единичная энергия.

    span — полуширина в символах, поэтому отводов 2·span·sps + 1.
    """
    t = np.arange(-span * sps, span * sps + 1) / sps
    h = np.zeros_like(t)
    for i, x in enumerate(t):
        if abs(x) < 1e-9:
            h[i] = 1 - alpha + 4 * alpha / np.pi
        elif abs(abs(x) - 1 / (4 * alpha)) < 1e-9:
            h[i] = alpha / np.sqrt(2) * (
                (1 + 2 / np.pi) * np.sin(np.pi / (4 * alpha))
                + (1 - 2 / np.pi) * np.cos(np.pi / (4 * alpha))
            )
        else:
            h[i] = (
                np.sin(np.pi * x * (1 - alpha))
                + 4 * alpha * x * np.cos(np.pi * x * (1 + alpha))
            ) / (np.pi * x * (1 - (4 * alpha * x) ** 2))
    return h / np.sqrt(np.sum(h**2))


def modulate(bits, mod):
    """
    Биты -> комплексные символы единичной энергии.

    QPSK: два бита на символ, код Грей (соседние точки различаются одним
    битом — при ошибке портится один бит, а не два).
    """
    if mod == "BPSK":
        return (1 - 2 * bits.astype(float)).astype(complex)
    b = bits[: len(bits) // 2 * 2].reshape(-1, 2)
    sym = (1 - 2 * b[:, 0]) + 1j * (1 - 2 * b[:, 1])
    return sym / np.sqrt(2)


def pulse_shape(sym, taps, sps):
    """Разложить символы по сетке с шагом sps и пропустить через фильтр."""
    up = np.zeros(len(sym) * sps, dtype=complex)
    up[::sps] = sym
    return np.convolve(up, taps, mode="full")

### 3. Канал

Шум добавляем по заданному Es/N0 — отношению энергии символа к спектральной
плотности шума. Оно не зависит от частоты дискретизации, поэтому результат
можно сравнивать с теорией напрямую.

Удобно запомнить: `Es/N0` и `Eb/N0` различаются на число бит в символе —
для QPSK это 3 дБ, для BPSK совпадают.

In [3]:
def add_noise(x, esn0_db, seed=CH_SEED):
    """
    Добавить комплексный гауссов шум под заданное Es/N0.

    Символы единичной энергии, поэтому мощность шума на отсчёт = 1/(Es/N0),
    то есть по половине на I и Q.
    """
    if esn0_db is None:
        return x
    esn0 = 10 ** (esn0_db / 10)
    sigma = np.sqrt(1.0 / (2.0 * esn0))
    rng = np.random.default_rng(seed)
    noise = sigma * (rng.standard_normal(x.size) + 1j * rng.standard_normal(x.size))
    return x + noise


def apply_cfo(x, fs, cfo_hz, phase_deg=0.0):
    """Расстройка несущей и постоянный набег фазы."""
    n = np.arange(x.size)
    return x * np.exp(1j * (2 * np.pi * cfo_hz * n / fs + np.deg2rad(phase_deg)))


def apply_timing(x, sps, err_symbols):
    """Сдвиг по времени на долю символа (дробная задержка через спектр)."""
    if err_symbols == 0:
        return x
    shift = err_symbols * sps
    n = np.fft.fftfreq(x.size) * x.size
    return np.fft.ifft(np.fft.fft(x) * np.exp(-2j * np.pi * n * shift / x.size))

### 4. Приёмник

Порядок обработки:

1. **Согласованный фильтр** — даёт максимум отношения сигнал/шум.
2. **Оценка частоты** — метод M-й степени: возведение в 4-ю степень (для
   QPSK) снимает модуляцию, в спектре появляется линия на `4·CFO`.
3. **Оценка такта** — метод Оэрдера-Мейера: у огибающей после фильтра есть
   периодичность с периодом символ; фаза этой периодичности даёт смещение.
4. **Подстройка фазы** — петля Костаса: ошибка по отклонению от ближайшей
   точки созвездия.

Важно: фильтр стоит первым. Оценка частоты по сырому сигналу при низком
отношении сигнал/шум срывается, а после фильтра выигрыш `sps` раз по шуму —
и оценка работает.

In [4]:
def estimate_cfo(x, fs, mod):
    """Грубая оценка расстройки частоты методом M-й степени."""
    p = 2 if mod == "BPSK" else 4
    z = x**p
    nfft = 1 << int(np.ceil(np.log2(z.size)))
    spec = np.abs(np.fft.fftshift(np.fft.fft(z, nfft)))
    freqs = np.fft.fftshift(np.fft.fftfreq(nfft, 1 / fs))
    k = int(np.argmax(spec))
    # параболическая интерполяция по трём точкам — точнее шага сетки
    if 0 < k < spec.size - 1:
        a, b, c = spec[k - 1], spec[k], spec[k + 1]
        d = 0.5 * (a - c) / (a - 2 * b + c + 1e-30)
    else:
        d = 0.0
    return (freqs[k] + d * (freqs[1] - freqs[0])) / p


def estimate_timing(y, sps):
    """Смещение стробирования (в отсчётах) методом Оэрдера-Мейера."""
    n = np.arange(y.size)
    z = np.abs(y) ** 2 * np.exp(-2j * np.pi * n / sps)
    m = y.size // sps
    z = z[: m * sps].reshape(m, sps).sum(axis=0)
    return -np.angle(z.sum()) / (2 * np.pi) * sps


def strobe(y, sps, offset, start):
    """
    Взять отсчёты в моменты стробирования.

    start — задержка фильтра: до неё сигнал ещё не набрался.
    offset — дробная поправка от оценки такта, в отсчётах.
    """
    idx = start + offset + np.arange(int((y.size - start - offset) / sps)) * sps
    idx = idx[(idx >= 0) & (idx < y.size - 1)]
    grid = np.arange(y.size)
    return np.interp(idx, grid, y.real) + 1j * np.interp(idx, grid, y.imag)


def costas(sym, mod, bw=0.005, damping=0.707):
    """
    Петля Костаса: подстройка фазы по решениям.

    Ошибка = Im(y · conj(d)), где d — принятое решение. Она равна нулю,
    когда созвездие стоит правильно.

    Начальную фазу берём грубо по самому сигналу: для QPSK у четвёртой
    степени всех точек одинаковый аргумент, поэтому `arg(mean(y⁴))/4` даёт
    сдвиг с точностью до 90°. Без этой оценки петля может «залипнуть» на
    повёрнутом созвездии — оно тоже даёт нулевую ошибку.

    bw — полоса петли в долях символьной скорости. Широкая петля быстрее
    втягивается, но сильнее шумит; 0.005 — разумный компромисс.
    """
    p = 2 if mod == "BPSK" else 4
    head = sym[: min(4000, sym.size)]
    phase = np.angle(np.mean(head**p)) / p - (0.0 if mod == "BPSK" else np.pi / 4)
    kp = 4 * damping * bw / (1 + 2 * damping * bw + bw**2)
    ki = 4 * bw**2 / (1 + 2 * damping * bw + bw**2)
    freq = 0.0
    out = np.empty_like(sym)
    for i, s in enumerate(sym):
        y = s * np.exp(-1j * phase)
        if mod == "BPSK":
            d = np.sign(y.real)
        else:
            d = np.sign(y.real) + 1j * np.sign(y.imag)
        err = (y * np.conj(d)).imag
        freq += ki * err
        phase += freq + kp * err
        out[i] = y
    return out


def demodulate(sym, mod):
    """Символы -> биты (обратное к модуляции)."""
    if mod == "BPSK":
        return (sym.real < 0).astype(int)
    bits = np.empty(sym.size * 2, dtype=int)
    bits[0::2] = sym.real < 0
    bits[1::2] = sym.imag < 0
    return bits


def count_errors(tx_bits, sym, mod):
    """
    Число ошибок с учётом неоднозначности фазы.

    Приёмник не знает абсолютной фазы: созвездие может оказаться повёрнутым
    на 90/180/270 градусов. Это не ошибка демодуляции, а неопределённость,
    которую снимают по преамбуле или маркеру кадра. Здесь проверяем все
    варианты и берём лучший.
    """
    best = None
    for k in range(2 if mod == "BPSK" else 4):
        rx = demodulate(sym * np.exp(-1j * np.pi / 2 * k), mod)
        for sh in range(3):
            a = tx_bits[sh:]
            n = min(a.size, rx.size)
            err = int(np.sum(a[:n] != rx[:n]))
            if best is None or err < best[0]:
                best = (err, n)
    return best


def receive(wave, taps, sps, mod, fs):
    """
    Весь приёмник: фильтр, частота, такт, фаза, решения.

    Возвращает символы, оценку частоты, оценку такта и выход фильтра
    (он нужен для графиков).
    """
    mf = np.convolve(wave, taps, mode="full")
    cfo = estimate_cfo(mf, fs, mod)
    corr = mf * np.exp(-2j * np.pi * cfo * np.arange(mf.size) / fs)
    tau = estimate_timing(corr, sps)
    sym = costas(strobe(corr, sps, tau, len(taps) - 1), mod)
    return sym, cfo, tau, mf

### 5. Прогон

Собираем тракт целиком и смотрим результат.

In [5]:
BPS = 1 if MOD == "BPSK" else 2
RS = RB / BPS              # символьная скорость, бод
FS = RS * SPS              # частота дискретизации, Гц
BW = RS * (1 + ALPHA)      # занимаемая полоса, Гц
TAPS = 2 * SPAN * SPS + 1  # отводов в фильтре

bits = make_bits(NBITS)
taps = rrc_taps(ALPHA, SPS, SPAN)
wave = pulse_shape(modulate(bits, MOD), taps, SPS)

# --- канал
wave = add_noise(wave, ESN0_DB)
if CFO_HZ or PHASE_DEG:
    wave = apply_cfo(wave, FS, CFO_HZ, PHASE_DEG)
if TIMING_ERR:
    wave = apply_timing(wave, SPS, TIMING_ERR)

# --- приёмник
sym_rx, cfo_est, tau, mf = receive(wave, taps, SPS, MOD, FS)
rx_bits = demodulate(sym_rx, MOD)
errors, n_checked = count_errors(bits, sym_rx, MOD)
ber_model = errors / n_checked

# --- теория
ebn0_db = ESN0_DB - 10 * np.log10(BPS) if ESN0_DB is not None else None
ber_theory = 0.5 * erfc(np.sqrt(10 ** (ebn0_db / 10))) if ebn0_db is not None else 0.0
evm = 100.0 / np.sqrt(10 ** (ESN0_DB / 10)) if ESN0_DB is not None else 0.0

rows = [
    ("Схема модуляции", f"{MOD}, {BPS} бит/символ"),
    ("Битовая скорость", f"{RB / 1e6:.3f} Мбит/с"),
    ("Символьная скорость", f"{RS / 1e6:.3f} Мбод"),
    ("Частота дискретизации", f"{FS / 1e6:.3f} МГц ({SPS} отсч./символ)"),
    ("Занимаемая полоса", f"{BW / 1e6:.3f} МГц (α = {ALPHA})"),
    ("Отводов в фильтре", f"{TAPS}"),
    ("Es/N0 задано", f"{ESN0_DB} дБ" if ESN0_DB is not None else "без шума"),
    ("Eb/N0 пересчёт", f"{ebn0_db:.2f} дБ" if ebn0_db is not None else "—"),
    ("Оценка CFO", f"{cfo_est:.1f} Гц (задано {CFO_HZ:.0f} Гц)"),
    ("Ошибка оценки CFO", f"{cfo_est - CFO_HZ:+.1f} Гц"),
    ("Оценка смещения такта", f"{tau:+.2f} отсчёта"),
    ("Ошибок / проверено бит", f"{errors} / {n_checked}"),
    ("BER модели", f"{ber_model:.3e}"),
    ("BER теории", f"{ber_theory:.3e}" if ebn0_db is not None else "—"),
    (
        "Отношение модель/теория",
        "—" if not ber_theory or errors == 0 else f"{ber_model / ber_theory:.2f}x",
    ),
    ("EVM", f"{evm:.1f} %"),
]
print("=" * 62)
print(f"РЕЗУЛЬТАТ: {MOD}, {RB / 1e6:.3f} Мбит/с, Es/N0 = {ESN0_DB} дБ")
print("=" * 62)
for name, value in rows:
    print(f"{name:32s} {value:>26s}")
print("=" * 62)
if ber_theory and errors == 0:
    print(f"Ошибок не найдено. По теории ожидалось около {ber_theory * n_checked:.1f} —")
    print("увеличьте NBITS, если нужна оценка на таком низком уровне ошибок")
if ber_theory and errors and ber_model / ber_theory < 1.5:
    print("Вывод: тракт работает, расхождение с теорией в пределах нормы")
elif ber_theory and errors:
    print("Вывод: есть потери — проверяйте синхронизацию и разрядности")

РЕЗУЛЬТАТ: QPSK, 1.000 Мбит/с, Es/N0 = 10.0 дБ
Схема модуляции                          QPSK, 2 бит/символ
Битовая скорость                               1.000 Мбит/с
Символьная скорость                              0.500 Мбод
Частота дискретизации            4.000 МГц (8 отсч./символ)
Занимаемая полоса                      0.675 МГц (α = 0.35)
Отводов в фильтре                                       129
Es/N0 задано                                        10.0 дБ
Eb/N0 пересчёт                                      6.99 дБ
Оценка CFO                            -0.0 Гц (задано 0 Гц)
Ошибка оценки CFO                                   -0.0 Гц
Оценка смещения такта                         -0.00 отсчёта
Ошибок / проверено бит                          91 / 100000
BER модели                                        9.100e-04
BER теории                                        7.827e-04
Отношение модель/теория                               1.16x
EVM                                                  

### 6. Спектр сигнала

Занимаемая полоса должна совпасть с расчётной `Rs·(1+α)`, а вне полосы
спектр должен быстро спадать.

In [6]:
def spectrum(x, fs, nfft=4096):
    """Спектр мощности в дБ с окном Ханна."""
    n = min(x.size, nfft)
    w = np.hanning(n)
    spec = np.fft.fftshift(np.fft.fft(x[:n] * w, nfft))
    freqs = np.fft.fftshift(np.fft.fftfreq(nfft, 1 / fs))
    return freqs, 20 * np.log10(np.abs(spec) / np.abs(spec).max() + 1e-12)


def thin(x, n=4000):
    """Прореживание для отображения (не для расчётов)."""
    if x.size <= n:
        return x
    return x[:: int(np.ceil(x.size / n))]


f, p = spectrum(wave, FS)
fig = go.Figure()
fig.add_trace(go.Scatter(x=f / 1e6, y=p, name="спектр",
                         line=dict(color="#38bdf8", width=1.5)))
for edge in (-BW / 2e6, BW / 2e6):
    fig.add_vline(x=edge, line_dash="dash", line_color="#fb7185")
fig.update_layout(
    title=f"Спектр сигнала: полоса {BW / 1e6:.3f} МГц",
    xaxis_title="Частота, МГц", yaxis_title="Уровень, дБ",
    template="plotly_dark", height=380,
)
fig.show()

### 7. Созвездие: до и после синхронизации

Главный инструмент отладки демодулятора. До синхронизации точки размазаны
и повёрнуты; после — должны собраться в кучки вокруг идеальных позиций.

In [7]:
before = strobe(mf, SPS, tau, len(taps) - 1)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=thin(before.real), y=thin(before.imag), mode="markers",
    name="до синхронизации",
    marker=dict(size=3, color="#fb7185", opacity=0.45),
))
fig.add_trace(go.Scatter(
    x=thin(sym_rx.real), y=thin(sym_rx.imag), mode="markers",
    name="после синхронизации",
    marker=dict(size=3, color="#34d399", opacity=0.55),
))
fig.update_layout(
    title="Созвездие: красное — до, зелёное — после",
    xaxis_title="I", yaxis_title="Q",
    xaxis=dict(scaleanchor="y", scaleratio=1, range=[-2, 2]),
    yaxis=dict(range=[-2, 2]),
    template="plotly_dark", height=480,
)
fig.show()

### 8. Глазковая диаграмма

Накладываем соседние отрезки по два символа. Раскрыв глаза показывает запас
по межсимвольной помехе: чем шире «зрачок», тем надёжнее стробирование.

In [8]:
seg = 2 * SPS
start = len(taps) - 1
n_seg = (mf.size - start) // seg
step = max(1, n_seg // 60)

fig = go.Figure()
for k in range(0, n_seg, step):
    chunk = mf[start + k * seg : start + (k + 1) * seg].real
    fig.add_trace(go.Scatter(
        x=np.arange(seg) / SPS, y=chunk, mode="lines",
        line=dict(color="#38bdf8", width=0.8), opacity=0.4, showlegend=False,
    ))
fig.update_layout(
    title="Глазковая диаграмма (по два символа)",
    xaxis_title="Время, символов", yaxis_title="Амплитуда",
    template="plotly_dark", height=400,
)
fig.show()

### 9. Кривая помехоустойчивости

Свип по отношению сигнал/шум: для каждой точки свой прогон со своим зерном
шума. Результат должен лечь на теоретическую кривую.

In [9]:
def run_point(esn0_db, nbits, mod, seed):
    """Один прогон тракта при заданном Es/N0 — число ошибок и проверенных бит."""
    b = make_bits(nbits, seed)
    w = pulse_shape(modulate(b, mod), taps, SPS)
    w = add_noise(w, esn0_db, seed=seed)
    if CFO_HZ or PHASE_DEG:
        w = apply_cfo(w, FS, CFO_HZ, PHASE_DEG)
    if TIMING_ERR:
        w = apply_timing(w, SPS, TIMING_ERR)
    r, _, _, _ = receive(w, taps, SPS, mod, FS)
    return count_errors(b, r, mod)


sweep_rows = []
for i, esn0 in enumerate(SWEEP):
    e, n = run_point(esn0, SWEEP_BITS, MOD, SEED + i)
    sweep_rows.append({
        "Es/N0, дБ": esn0,
        "Eb/N0, дБ": esn0 - 10 * np.log10(BPS),
        "ошибок": e,
        "бит": n,
        "BER модели": e / n,
    })

df = pd.DataFrame(sweep_rows)
ebn0_grid = np.linspace(min(SWEEP) - 2, max(SWEEP) + 1, 200)
theory = 0.5 * erfc(np.sqrt(10 ** ((ebn0_grid - 10 * np.log10(BPS)) / 10)))

fig = go.Figure()
fig.add_trace(go.Scatter(x=ebn0_grid, y=theory, name="теория", mode="lines",
                         line=dict(color="#fb7185", width=2)))
fig.add_trace(go.Scatter(x=df["Eb/N0, дБ"], y=df["BER модели"], name="модель",
                         mode="lines+markers", line=dict(color="#34d399", width=2),
                         marker=dict(size=8)))
fig.update_layout(
    title="Помехоустойчивость: модель против теории",
    xaxis_title="Eb/N0, дБ", yaxis_title="Вероятность ошибки на бит",
    yaxis=dict(type="log", range=[-6, 0]),
    template="plotly_dark", height=440,
)
fig.show()
print(df.to_string(index=False, float_format=lambda v: f"{v:.3e}"))

 Es/N0, дБ  Eb/N0, дБ  ошибок   бит  BER модели
         4  9.897e-01    2352 40000   5.880e-02
         6  2.990e+00     914 40000   2.285e-02
         8  4.990e+00     247 40000   6.175e-03
        10  6.990e+00      36 40000   9.000e-04
        12  8.990e+00       2 40000   5.000e-05


### 10. Фиксированная точка

То, что будет в цифровой реализации: коэффициенты и данные хранятся с
ограниченной разрядностью. Считаем, насколько от этого падает точность.

Правило: `SQNR = 6.02·wl + 1.76 дБ`. Если SQNR заметно выше рабочего
отношения сигнал/шум — разрядности достаточно.

In [10]:
def to_fixed(x, wl, fl):
    """Квантование в формат с fl дробными битами и насыщением."""
    scale = 2.0**fl
    lo, hi = -(2 ** (wl - 1)), 2 ** (wl - 1) - 1
    if np.iscomplexobj(x):
        re = np.clip(np.round(x.real * scale), lo, hi).astype(np.int64)
        im = np.clip(np.round(x.imag * scale), lo, hi).astype(np.int64)
        return re + 1j * im
    return np.clip(np.round(x * scale), lo, hi).astype(np.int64)


def from_fixed(ints, fl):
    """Обратное преобразование."""
    return ints / 2.0**fl


COEFF_WL, COEFF_FL = 16, 14
DATA_WL, DATA_FL = 16, 13

taps_q = from_fixed(to_fixed(taps, COEFF_WL, COEFF_FL), COEFF_FL)
wave_q = from_fixed(to_fixed(wave, DATA_WL, DATA_FL), DATA_FL)
sym_q, _, _, _ = receive(wave_q, taps_q, SPS, MOD, FS)
err_q, n_q = count_errors(bits, sym_q, MOD)

print(f"Коэффициенты: Q{COEFF_WL - COEFF_FL}.{COEFF_FL}, SQNR ≈ {6.02 * COEFF_WL + 1.76:.1f} дБ")
print(f"Данные:       Q{DATA_WL - DATA_FL}.{DATA_FL}, SQNR ≈ {6.02 * DATA_WL + 1.76:.1f} дБ")
print(f"Рабочее Es/N0: {ESN0_DB} дБ")
print()
print(f"BER в плавающей точке: {ber_model:.3e}")
print(f"BER в фиксированной:   {err_q / n_q:.3e}")
print()
if 6.02 * DATA_WL + 1.76 > ESN0_DB + 10:
    print("Вывод: разрядностей достаточно, квантование не портит результат")
else:
    print("Вывод: стоит увеличить разрядность данных")

Коэффициенты: Q2.14, SQNR ≈ 98.1 дБ
Данные:       Q3.13, SQNR ≈ 98.1 дБ
Рабочее Es/N0: 10.0 дБ

BER в плавающей точке: 9.100e-04
BER в фиксированной:   9.200e-04

Вывод: разрядностей достаточно, квантование не портит результат


### 11. Экспорт для ПЛИС

Что нужно, чтобы начать реализацию: файл коэффициентов и тест-векторы для
сверки железа с моделью. Формат `.coe` — стандартный для Xilinx.

In [11]:
out = Path("out")
out.mkdir(exist_ok=True)


def save_coe(path, values, wl, fl):
    """Коэффициенты в формате .coe (Xilinx)."""
    ints = to_fixed(values, wl, fl)
    body = ",\n".join(f"{int(v) & ((1 << wl) - 1):0{wl // 4}x}" for v in ints)
    Path(path).write_text(
        "memory_initialization_radix=16;\nmemory_initialization_vector=\n" + body + ";",
        encoding="utf-8",
    )
    return Path(path)


def save_hex(path, values, wl, fl):
    """Вектор в hex, по значению на строку."""
    ints = to_fixed(values, wl, fl)
    mask = (1 << wl) - 1
    lines = [f"{int(np.real(v)) & mask:0{wl // 4}x}" for v in np.atleast_1d(ints)]
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")
    return Path(path)


files = [
    save_coe(out / "rrc_taps.coe", taps, COEFF_WL, COEFF_FL),
    save_hex(out / "fir_input.hex", wave.real[:2000], DATA_WL, DATA_FL),
    save_hex(out / "fir_output.hex", mf.real[:2000], DATA_WL, DATA_FL),
]
print("Сохранено:")
for p in files:
    print(f"  {p}  ({p.stat().st_size} байт)")

acc_wl = DATA_WL + COEFF_WL + int(np.ceil(np.log2(TAPS)))
print()
print(f"Разрядность накопителя: {DATA_WL} + {COEFF_WL} + log2({TAPS}) = {acc_wl} бит")
print(f"Умножителей DSP48 (симметричный КИХ): {(TAPS + 1) // 2}")
print(f"Тактовая частота обработки: не ниже {FS / 1e6:.2f} МГц")

Сохранено:
  out\rrc_taps.coe  (965 байт)
  out\fir_input.hex  (12000 байт)


  out\fir_output.hex  (12000 байт)

Разрядность накопителя: 16 + 16 + log2(129) = 40 бит
Умножителей DSP48 (симметричный КИХ): 65
Тактовая частота обработки: не ниже 4.00 МГц


### 12. Что дальше

Шпаргалка по параметрам:

| Что нужно | Что менять | На что смотреть |
|---|---|---|
| другая скорость | `RB` | полоса, `FS`, ресурсы |
| другая модуляция | `MOD` | BER, пик-фактор |
| уже полоса | `ALPHA` меньше | длина фильтра, чувствительность к такту |
| хуже канал | `ESN0_DB` меньше | BER, форма кривой |
| есть расстройка | `CFO_HZ` | оценка CFO, созвездие |
| реальный АЦП | `DATA_WL` | деградация от квантования |

Если результат не сходится с теорией — порядок разбора:

1. Выключить шум (`ESN0_DB = None`): ошибок должно быть 0.
2. Включить только шум: BER должен совпасть с теорией.
3. Включать искажения по одному и смотреть, после какого ломается.
4. Проверить квантование.
5. Смотреть созвездие: оно говорит правду быстрее цифр.